# Finite-horizon LQR and the Riccati recursion

**Learning goals:** run the backward Riccati recursion, apply the resulting feedback gains, and inspect closed-loop state decay.

**Predict first:** increasing the control penalty should make the feedback gains larger or smaller?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider

A = np.array([[1.0, 0.1], [0.0, 1.0]])
B = np.array([[0.005], [0.1]])
Q = np.diag([1.0, 0.1])
Q_terminal = 10 * Q
horizon = 60

def riccati(control_penalty=0.1):
    R = np.array([[control_penalty]])
    P = [None] * (horizon + 1)
    gains = [None] * horizon
    P[-1] = Q_terminal
    for k in range(horizon - 1, -1, -1):
        gains[k] = np.linalg.solve(R + B.T @ P[k + 1] @ B, B.T @ P[k + 1] @ A)
        P[k] = Q + A.T @ P[k + 1] @ (A - B @ gains[k])
    return P, gains

def rollout(control_penalty=0.1):
    P, gains = riccati(control_penalty)
    states = np.zeros((horizon + 1, 2))
    controls = np.zeros(horizon)
    states[0] = [2.0, 0.0]
    for k in range(horizon):
        controls[k] = (-gains[k] @ states[k]).item()
        states[k + 1] = A @ states[k] + B[:, 0] * controls[k]
    return P, gains, states, controls

In [ ]:
@interact(control_penalty=FloatLogSlider(value=0.1, base=10, min=-2, max=1, step=0.25))
def lqr_plot(control_penalty):
    P, gains, states, controls = rollout(control_penalty)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    axes[0].plot(states[:, 0], label="position")
    axes[0].plot(states[:, 1], label="velocity")
    axes[0].legend()
    axes[1].step(np.arange(horizon), controls, where="post")
    axes[1].set_title(f"First gain: {gains[0].ravel()}")
    fig.tight_layout()
    plt.show()

**Experiment:** increase the terminal cost by a factor of ten. Which gains change most: those near the beginning or those near the end of the horizon?

In [ ]:
P, gains, states, controls = rollout()
assert all(np.allclose(matrix, matrix.T, atol=1e-10) for matrix in P)
assert np.linalg.norm(states[-1]) < np.linalg.norm(states[0])
assert np.all(np.linalg.eigvalsh(P[0]) > 0)
print("Checks passed.")